# LSTM Neural Network Model (PyTorch)

Build a deep learning sentiment classifier using PyTorch LSTM. This demonstrates neural network expertise with a manual training loop, custom vocabulary building, and sequence modeling for NLP.

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter
import re
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize
import joblib
import os
import io
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('✓ All imports successful')

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


✓ All imports successful


## Helper Classes

In [2]:
class SentimentLSTM(nn.Module):
    """PyTorch LSTM for 3-class sentiment classification."""
    def __init__(self, int_vocab_size, int_embedding_dim, int_lstm_units, int_num_classes, flt_dropout):
        super().__init__()
        self.embedding = nn.Embedding(int_vocab_size, int_embedding_dim, padding_idx=0)
        self.spatial_dropout = nn.Dropout2d(flt_dropout)
        self.lstm = nn.LSTM(int_embedding_dim, int_lstm_units, batch_first=True, dropout=flt_dropout)
        self.fc1 = nn.Linear(int_lstm_units, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(flt_dropout)
        self.fc2 = nn.Linear(32, int_num_classes)
    
    def forward(self, x):
        # Embedding layer
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        # SpatialDropout1D equivalent: dropout on embedding dimension
        embedded = self.spatial_dropout(embedded.unsqueeze(3)).squeeze(3)
        # LSTM layer
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Use final hidden state
        out = self.fc1(hidden[-1])
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

print('✓ SentimentLSTM class defined')

✓ SentimentLSTM class defined


## Constants

In [3]:
str_bucket = 'nlp-sentiment-analysis-demo'
str_dirname_output = './output'
int_max_words = 5000
int_max_len = 100
int_embedding_dim = 64
int_lstm_units = 64
flt_dropout = 0.3
int_epochs = 10
int_batch_size = 128
int_patience = 3
int_num_classes = 3

print('✓ Constants defined')

✓ Constants defined


## Output Directory

In [4]:
if not os.path.exists(str_dirname_output):
    os.mkdir(str_dirname_output)
    print(f'✓ Created output directory: {str_dirname_output}')
else:
    print(f'✓ Output directory already exists: {str_dirname_output}')

✓ Output directory already exists: ./output


## LSTMSentimentModel Class

In [5]:
class LSTMSentimentModel:
    """Complete LSTM sentiment analysis model with PyTorch."""
    
    def __init__(self, str_bucket, str_dirname_output):
        """Initialize S3 client, detect device, set plot style."""
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        sns.set_style('whitegrid')
        
        # Data containers
        self.df_train = None
        self.df_val = None
        self.df_test = None
        self.dict_word2idx = {}
        self.arr_train_sequences = None
        self.arr_val_sequences = None
        self.arr_test_sequences = None
        self.arr_train_labels = None
        self.arr_val_labels = None
        self.arr_test_labels = None
        self.train_loader = None
        self.val_loader = None
        self.test_loader = None
        self.model = None
        self.criterion = None
        self.optimizer = None
        self.dict_history = {}
        
        print(f'LSTMSentimentModel initialized on device: {self.device}')
    
    def import_data(self):
        """Load train/val/test from S3."""
        list_splits = [
            ('train_split.parquet', 'df_train'),
            ('validation_split.parquet', 'df_val'),
            ('test_split.parquet', 'df_test')
        ]
        
        for str_filename, str_attr in list_splits:
            str_uri = f's3://{self.str_bucket}/02_preprocessing/{str_filename}'
            df_temp = pd.read_parquet(str_uri)
            setattr(self, str_attr, df_temp)
            print(f'Loaded {str_filename} from S3 ({len(df_temp)} rows)')
    
    def build_vocabulary(self, int_max_words=20000):
        """Build word-to-index mapping from training text."""
        list_words = []
        for str_text in self.df_train['review_text_clean']:
            list_words.extend(str(str_text).split())
        
        obj_counter = Counter(list_words)
        list_top_words = [word for word, _ in obj_counter.most_common(int_max_words)]
        
        self.dict_word2idx = {'<PAD>': 0, '<UNK>': 1}
        for int_idx, str_word in enumerate(list_top_words, start=2):
            self.dict_word2idx[str_word] = int_idx
        
        int_vocab_size = len(self.dict_word2idx)
        print(f'Vocabulary built with {int_vocab_size} words (PAD=0, UNK=1, top {int_max_words} words)')
    
    def texts_to_sequences(self, list_texts, int_max_len=200):
        """Convert list of text strings to padded integer sequences."""
        list_sequences = []
        for str_text in list_texts:
            list_words = str(str_text).split()
            list_seq = []
            for str_word in list_words:
                int_idx = self.dict_word2idx.get(str_word, 1)  # 1 is <UNK>
                list_seq.append(int_idx)
            
            # Truncate or pad
            if len(list_seq) > int_max_len:
                list_seq = list_seq[:int_max_len]
            else:
                list_seq = list_seq + [0] * (int_max_len - len(list_seq))
            
            list_sequences.append(list_seq)
        
        arr_sequences = np.array(list_sequences, dtype=np.int64)
        return torch.LongTensor(arr_sequences)
    
    def prepare_dataloaders(self, int_batch_size=64):
        """Create torch DataLoaders for train/val/test."""
        # Convert texts to sequences
        self.arr_train_sequences = self.texts_to_sequences(self.df_train['review_text_clean'].values, int_max_len)
        self.arr_val_sequences = self.texts_to_sequences(self.df_val['review_text_clean'].values, int_max_len)
        self.arr_test_sequences = self.texts_to_sequences(self.df_test['review_text_clean'].values, int_max_len)
        
        # Extract labels
        self.arr_train_labels = torch.LongTensor(self.df_train['sentiment_3class'].values)
        self.arr_val_labels = torch.LongTensor(self.df_val['sentiment_3class'].values)
        self.arr_test_labels = torch.LongTensor(self.df_test['sentiment_3class'].values)
        
        # Create TensorDatasets
        obj_train_dataset = TensorDataset(self.arr_train_sequences, self.arr_train_labels)
        obj_val_dataset = TensorDataset(self.arr_val_sequences, self.arr_val_labels)
        obj_test_dataset = TensorDataset(self.arr_test_sequences, self.arr_test_labels)
        
        # Create DataLoaders
        self.train_loader = DataLoader(obj_train_dataset, batch_size=int_batch_size, shuffle=True)
        self.val_loader = DataLoader(obj_val_dataset, batch_size=int_batch_size, shuffle=False)
        self.test_loader = DataLoader(obj_test_dataset, batch_size=int_batch_size, shuffle=False)
        
        print(f'DataLoaders created (batch_size={int_batch_size})')
        print(f'  Train: {len(self.train_loader)} batches')
        print(f'  Val: {len(self.val_loader)} batches')
        print(f'  Test: {len(self.test_loader)} batches')
    
    def build_model(self, int_embedding_dim=128, int_lstm_units=64, flt_dropout=0.3):
        """Define and instantiate SentimentLSTM model."""
        int_vocab_size = len(self.dict_word2idx)
        self.model = SentimentLSTM(
            int_vocab_size=int_vocab_size,
            int_embedding_dim=int_embedding_dim,
            int_lstm_units=int_lstm_units,
            int_num_classes=int_num_classes,
            flt_dropout=flt_dropout
        ).to(self.device)
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=1e-3)
        
        # Print model summary
        int_total_params = sum(p.numel() for p in self.model.parameters())
        int_trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f'Model built on {self.device}')
        print(f'  Total parameters: {int_total_params:,}')
        print(f'  Trainable parameters: {int_trainable_params:,}')
        print(f'  Embedding dim: {int_embedding_dim}, LSTM units: {int_lstm_units}, Dropout: {flt_dropout}')
    
    def train_model(self, int_epochs=20, int_patience=3):
        """Manual training loop with early stopping and learning rate reduction."""
        self.dict_history = {
            'train_loss': [],
            'val_loss': [],
            'train_acc': [],
            'val_acc': []
        }
        
        obj_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=2, verbose=False
        )
        
        flt_best_val_loss = float('inf')
        int_patience_counter = 0
        
        for int_epoch in range(int_epochs):
            # Training phase
            self.model.train()
            flt_train_loss = 0.0
            int_train_correct = 0
            int_train_total = 0
            
            for arr_batch_x, arr_batch_y in self.train_loader:
                arr_batch_x = arr_batch_x.to(self.device)
                arr_batch_y = arr_batch_y.to(self.device)
                
                self.optimizer.zero_grad()
                arr_logits = self.model(arr_batch_x)
                flt_loss = self.criterion(arr_logits, arr_batch_y)
                flt_loss.backward()
                self.optimizer.step()
                
                flt_train_loss += flt_loss.item() * arr_batch_x.size(0)
                _, arr_preds = torch.max(arr_logits, 1)
                int_train_correct += (arr_preds == arr_batch_y).sum().item()
                int_train_total += arr_batch_y.size(0)
            
            flt_train_loss /= int_train_total
            flt_train_acc = int_train_correct / int_train_total
            
            # Validation phase
            self.model.eval()
            flt_val_loss = 0.0
            int_val_correct = 0
            int_val_total = 0
            
            with torch.no_grad():
                for arr_batch_x, arr_batch_y in self.val_loader:
                    arr_batch_x = arr_batch_x.to(self.device)
                    arr_batch_y = arr_batch_y.to(self.device)
                    
                    arr_logits = self.model(arr_batch_x)
                    flt_loss = self.criterion(arr_logits, arr_batch_y)
                    
                    flt_val_loss += flt_loss.item() * arr_batch_x.size(0)
                    _, arr_preds = torch.max(arr_logits, 1)
                    int_val_correct += (arr_preds == arr_batch_y).sum().item()
                    int_val_total += arr_batch_y.size(0)
            
            flt_val_loss /= int_val_total
            flt_val_acc = int_val_correct / int_val_total
            
            # Store history
            self.dict_history['train_loss'].append(flt_train_loss)
            self.dict_history['val_loss'].append(flt_val_loss)
            self.dict_history['train_acc'].append(flt_train_acc)
            self.dict_history['val_acc'].append(flt_val_acc)
            
            # Print progress
            print(f'Epoch {int_epoch+1}/{int_epochs} | Train Loss: {flt_train_loss:.4f} | Train Acc: {flt_train_acc:.4f} | Val Loss: {flt_val_loss:.4f} | Val Acc: {flt_val_acc:.4f}')
            
            # Early stopping and LR reduction
            if flt_val_loss < flt_best_val_loss:
                flt_best_val_loss = flt_val_loss
                int_patience_counter = 0
                # Save best model
                self.dict_best_state = self.model.state_dict().copy()
            else:
                int_patience_counter += 1
            
            obj_scheduler.step(flt_val_loss)
            
            if int_patience_counter >= int_patience:
                print(f'Early stopping at epoch {int_epoch+1}')
                break
        
        # Load best model weights
        self.model.load_state_dict(self.dict_best_state)
        print(f'Model training completed, best val loss: {flt_best_val_loss:.4f}')
    
    def plot_training_history(self):
        """Plot training history (loss and accuracy)."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
        
        # Loss
        ax1.plot(self.dict_history['train_loss'], label='Train Loss', marker='o', linewidth=2)
        ax1.plot(self.dict_history['val_loss'], label='Val Loss', marker='s', linewidth=2)
        ax1.set_xlabel('Epoch', fontsize=12)
        ax1.set_ylabel('Loss', fontsize=12)
        ax1.set_title('Training History - Loss', fontsize=13, fontweight='bold')
        ax1.legend(fontsize=10)
        ax1.grid(True, alpha=0.3)
        
        # Accuracy
        ax2.plot(self.dict_history['train_acc'], label='Train Accuracy', marker='o', linewidth=2)
        ax2.plot(self.dict_history['val_acc'], label='Val Accuracy', marker='s', linewidth=2)
        ax2.set_xlabel('Epoch', fontsize=12)
        ax2.set_ylabel('Accuracy', fontsize=12)
        ax2.set_title('Training History - Accuracy', fontsize=13, fontweight='bold')
        ax2.legend(fontsize=10)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        str_filepath = f'{self.str_dirname_output}/01_training_history.png'
        plt.savefig(str_filepath, bbox_inches='tight', dpi=150)
        print(f'Training history plot saved: {str_filepath}')
        plt.close()
    
    def evaluate(self):
        """Evaluate model on test set and compute metrics."""
        self.model.eval()
        list_y_pred = []
        list_y_proba = []
        list_y_true = []
        
        with torch.no_grad():
            for arr_batch_x, arr_batch_y in self.test_loader:
                arr_batch_x = arr_batch_x.to(self.device)
                arr_logits = self.model(arr_batch_x)
                arr_proba = torch.softmax(arr_logits, dim=1)
                
                _, arr_preds = torch.max(arr_logits, 1)
                list_y_pred.extend(arr_preds.cpu().numpy())
                list_y_proba.extend(arr_proba.cpu().numpy())
                list_y_true.extend(arr_batch_y.numpy())
        
        arr_y_pred = np.array(list_y_pred)
        arr_y_proba = np.array(list_y_proba)
        arr_y_true = np.array(list_y_true)
        
        flt_accuracy = accuracy_score(arr_y_true, arr_y_pred)
        flt_f1_macro = f1_score(arr_y_true, arr_y_pred, average='macro')
        flt_f1_weighted = f1_score(arr_y_true, arr_y_pred, average='weighted')
        
        try:
            flt_roc_auc = roc_auc_score(arr_y_true, arr_y_proba, multi_class='ovr', average='macro')
        except:
            flt_roc_auc = 0.0
        
        print('\n' + '='*60)
        print('Test Set Evaluation')
        print('='*60)
        print(f'Accuracy:      {flt_accuracy:.4f}')
        print(f'F1 (Macro):    {flt_f1_macro:.4f}')
        print(f'F1 (Weighted): {flt_f1_weighted:.4f}')
        print(f'ROC AUC (OvR): {flt_roc_auc:.4f}')
        print('\nClassification Report:')
        print(classification_report(arr_y_true, arr_y_pred, target_names=['Negative', 'Neutral', 'Positive']))
        print('='*60 + '\n')
        
        return arr_y_pred, arr_y_proba, arr_y_true
    
    def plot_confusion_matrix(self, arr_y_pred, arr_y_true):
        """Plot confusion matrix."""
        arr_cm = confusion_matrix(arr_y_true, arr_y_pred)
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(arr_cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                    xticklabels=['Negative', 'Neutral', 'Positive'],
                    yticklabels=['Negative', 'Neutral', 'Positive'],
                    annot_kws={'size': 12})
        plt.title('Confusion Matrix - LSTM Neural Network (PyTorch)', fontsize=13, fontweight='bold')
        plt.ylabel('True Label', fontsize=12)
        plt.xlabel('Predicted Label', fontsize=12)
        plt.tight_layout()
        
        str_filepath = f'{self.str_dirname_output}/02_confusion_matrix.png'
        plt.savefig(str_filepath, bbox_inches='tight', dpi=150)
        print(f'Confusion matrix plot saved: {str_filepath}')
        plt.close()
    
    def plot_roc_curves(self, arr_y_proba, arr_y_true):
        """Plot ROC curves for 3-class sentiment."""
        arr_y_bin = label_binarize(arr_y_true, classes=[0, 1, 2])
        list_colors = ['#d62728', '#ff7f0e', '#2ca02c']
        list_class_labels = ['Negative', 'Neutral', 'Positive']
        
        plt.figure(figsize=(10, 8))
        for int_i in range(int_num_classes):
            arr_fpr, arr_tpr, _ = roc_curve(arr_y_bin[:, int_i], arr_y_proba[:, int_i])
            flt_auc = auc(arr_fpr, arr_tpr)
            plt.plot(arr_fpr, arr_tpr, color=list_colors[int_i], lw=2,
                    label=f'{list_class_labels[int_i]} (AUC = {flt_auc:.3f})')
        
        plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.title('ROC Curves - LSTM Neural Network (PyTorch)', fontsize=13, fontweight='bold')
        plt.legend(loc='lower right', fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        str_filepath = f'{self.str_dirname_output}/03_roc_curves.png'
        plt.savefig(str_filepath, bbox_inches='tight', dpi=150)
        print(f'ROC curves plot saved: {str_filepath}')
        plt.close()
    
    def save_model(self):
        """Save model state, vocabulary, and configuration."""
        # Save model state dict
        str_model_path = f'{self.str_dirname_output}/lstm_model.pt'
        torch.save(self.model.state_dict(), str_model_path)
        print(f'Model state dict saved: {str_model_path}')
        
        # Save vocabulary
        str_vocab_path = f'{self.str_dirname_output}/vocabulary.pkl'
        joblib.dump(self.dict_word2idx, str_vocab_path)
        print(f'Vocabulary saved: {str_vocab_path}')
        
        # Save configuration
        dict_config = {
            'int_vocab_size': len(self.dict_word2idx),
            'int_max_len': int_max_len,
            'int_embedding_dim': int_embedding_dim,
            'int_lstm_units': int_lstm_units,
            'flt_dropout': flt_dropout,
            'int_num_classes': int_num_classes
        }
        str_config_path = f'{self.str_dirname_output}/model_config.pkl'
        joblib.dump(dict_config, str_config_path)
        print(f'Model config saved: {str_config_path}')

print('LSTMSentimentModel class defined')

LSTMSentimentModel class defined


## Load Data and Prepare Sequences

In [6]:
obj_model = LSTMSentimentModel(str_bucket, str_dirname_output)
obj_model.import_data()
obj_model.build_vocabulary(int_max_words)
obj_model.prepare_dataloaders(int_batch_size)

LSTMSentimentModel initialized on device: cpu
Loaded train_split.parquet from S3 (18168 rows)
Loaded validation_split.parquet from S3 (6056 rows)
Loaded test_split.parquet from S3 (6057 rows)
Vocabulary built with 5002 words (PAD=0, UNK=1, top 5000 words)
DataLoaders created (batch_size=128)
  Train: 142 batches
  Val: 48 batches
  Test: 48 batches


## Build and Train Model

In [7]:
obj_model.build_model(int_embedding_dim, int_lstm_units, flt_dropout)
obj_model.train_model(int_epochs, int_patience)

Model built on cpu
  Total parameters: 355,587
  Trainable parameters: 355,587
  Embedding dim: 64, LSTM units: 64, Dropout: 0.3
Epoch 1/10 | Train Loss: 0.9869 | Train Acc: 0.5342 | Val Loss: 0.9692 | Val Acc: 0.5495
Epoch 2/10 | Train Loss: 0.9669 | Train Acc: 0.5495 | Val Loss: 0.9641 | Val Acc: 0.5497
Epoch 3/10 | Train Loss: 0.9673 | Train Acc: 0.5500 | Val Loss: 0.9629 | Val Acc: 0.5500
Epoch 4/10 | Train Loss: 0.9646 | Train Acc: 0.5514 | Val Loss: 0.9639 | Val Acc: 0.5502
Epoch 5/10 | Train Loss: 0.9620 | Train Acc: 0.5538 | Val Loss: 0.9636 | Val Acc: 0.5502
Epoch 6/10 | Train Loss: 0.9584 | Train Acc: 0.5559 | Val Loss: 0.9645 | Val Acc: 0.5497
Early stopping at epoch 6
Model training completed, best val loss: 0.9629


## Training History

In [8]:
obj_model.plot_training_history()

Training history plot saved: ./output/01_training_history.png


## Evaluate Model

In [9]:
arr_y_pred, arr_y_proba, arr_y_true = obj_model.evaluate()
obj_model.plot_confusion_matrix(arr_y_pred, arr_y_true)
obj_model.plot_roc_curves(arr_y_proba, arr_y_true)


Test Set Evaluation
Accuracy:      0.5476
F1 (Macro):    0.2542
F1 (Weighted): 0.3988
ROC AUC (OvR): 0.5085

Classification Report:
              precision    recall  f1-score   support

    Negative       0.28      0.02      0.04       830
     Neutral       0.25      0.01      0.01      1899
    Positive       0.55      0.99      0.71      3328

    accuracy                           0.55      6057
   macro avg       0.36      0.34      0.25      6057
weighted avg       0.42      0.55      0.40      6057


Confusion matrix plot saved: ./output/02_confusion_matrix.png
ROC curves plot saved: ./output/03_roc_curves.png


## Save Model

In [10]:
obj_model.save_model()

Model state dict saved: ./output/lstm_model.pt
Vocabulary saved: ./output/vocabulary.pkl
Model config saved: ./output/model_config.pkl


## Summary

In [11]:
print('\n' + '='*60)
print('LSTM Neural Network Training Complete')
print('='*60)
print(f'✓ Model: SentimentLSTM (PyTorch)')
print(f'✓ Device: {obj_model.device}')
print(f'✓ Vocabulary size: {len(obj_model.dict_word2idx)}')
print(f'✓ Sequence length: {int_max_len}')
print(f'✓ Embedding dimension: {int_embedding_dim}')
print(f'✓ LSTM units: {int_lstm_units}')
print(f'✓ Dropout: {flt_dropout}')
print(f'✓ Training epochs: {len(obj_model.dict_history["train_loss"])}')
print(f'✓ Output directory: {str_dirname_output}')
print('='*60)


LSTM Neural Network Training Complete
✓ Model: SentimentLSTM (PyTorch)
✓ Device: cpu
✓ Vocabulary size: 5002
✓ Sequence length: 100
✓ Embedding dimension: 64
✓ LSTM units: 64
✓ Dropout: 0.3
✓ Training epochs: 6
✓ Output directory: ./output
